# Clase 10 — VQE avanzado: Rotosolve y comparativa de optimizadores

## Objetivos de aprendizaje

- Implementar Rotosolve y explicar por qué funciona sobre puertas de rotación
- Comparar Rotosolve, Nelder-Mead y SPSA sobre el mismo problema

En la clase anterior optimizaste VQE con un método basado en gradiente y con Nelder-Mead, un optimizador de caja negra genérico. En esta clase conoces Rotosolve, un optimizador diseñado específicamente para circuitos cuánticos parametrizados con rotaciones, y lo comparas con Nelder-Mead y SPSA sobre el mismo problema.

In [ ]:
!pip install -q cirq scipy noisyopt

## Por qué importa

En el capítulo anterior probaste dos optimizadores para VQE: un método basado en gradiente (Adam) y un optimizador de caja negra genérico (Nelder-Mead). Ninguno de los dos aprovecha una propiedad matemática exclusiva de las puertas de rotación cuántica. Rotosolve sí la aprovecha: en cada iteración encuentra el valor óptimo exacto de un parámetro, sin avanzar en pasos pequeños de gradiente ni adivinar por prueba y error. Entender Rotosolve te da intuición sobre cómo diseñar optimizadores a medida para problemas cuánticos, en vez de aplicar siempre las mismas herramientas clásicas.

## Por qué un parámetro de rotación se comporta como una onda

Fija todos los parámetros de tu ansatz salvo uno, y deja que ese ángulo de una puerta `rx`, `ry` o `rz` varíe libremente. La energía resultante `E(θ)` no es una curva cualquiera: siempre tiene la forma `E(θ) = A·cos(θ) + B·sin(θ) + C`, una onda sinusoidal simple. Esta propiedad es consecuencia directa de cómo las puertas de rotación transforman el estado cuántico. Ya lo comprobaste en la clase anterior: la curva de energía que dibujaste al barrer el ángulo de cero a dos π tenía justo esa forma ondulada.

## Rotosolve: resolver la onda en vez de aproximarla

Si sabes que `E(θ)` es una onda con esa forma exacta, no necesitas dar pasitos de gradiente para encontrar su mínimo. Te basta con evaluar la energía en tres puntos concretos (un valor `φ` cualquiera, y `φ + π/2` y `φ - π/2`) para despejar los coeficientes A, B y C, y calcular directamente el ángulo que minimiza la onda. Rotosolve repite este proceso parámetro a parámetro, en pasadas sucesivas: en cada pasada ajusta cada uno a su valor óptimo exacto suponiendo que el resto se mantiene fijo. Tras varias pasadas, el conjunto completo converge.

La ventaja frente a un método basado en gradiente: cada actualización es un salto directo al óptimo (para ese parámetro, con los demás fijos), no un paso pequeño en la dirección de mayor pendiente. La desventaja: necesitas evaluar el circuito varias veces por cada parámetro y por cada pasada, así que el coste por iteración es mayor.

## Comparando tres optimizadores en igualdad de condiciones

Además de Rotosolve, esta clase compara dos optimizadores clásicos de caja negra:

- **Nelder-Mead**: explora el espacio de parámetros con un "simplex" (una figura geométrica que se deforma) que se contrae hacia la región de menor energía; es robusto, pero puede necesitar muchas evaluaciones cuando el ansatz tiene muchos parámetros
- **SPSA** (aproximación estocástica por perturbación simultánea): en cada paso perturba todos los parámetros a la vez con una dirección aleatoria, evalúa la energía en esa dirección y estima el gradiente completo con solo dos evaluaciones del circuito, sea cual sea su número total. Es útil cuando evaluar el circuito es caro (por ejemplo, en hardware cuántico real) y tienes muchos parámetros que optimizar

Comparar los tres optimizadores sobre el mismo problema te muestra que ninguno es universalmente mejor: la elección depende de cuántos parámetros tengas, de cuánto cueste evaluar el circuito y de cuánto ruido haya en esa evaluación.

## Primer ejemplo: Rotosolve reducido

A continuación tienes una versión reducida de Rotosolve, con solo 30 iteraciones para que se ejecute en pocos segundos. Recibe una función de energía cualquiera y el número de parámetros, y devuelve el ángulo óptimo encontrado para cada uno.

In [ ]:
import numpy as np
import random

def rotosolve(funcion_energia, num_parametros, iteraciones=30):
    """Optimiza cada parámetro de forma analítica, ciclo a ciclo."""
    theta = np.random.uniform(-np.pi, np.pi, num_parametros)
    for _ in range(iteraciones):
        for d in range(num_parametros):
            phi = random.uniform(-np.pi, np.pi)
            theta_prueba = np.copy(theta)

            theta_prueba[d] = phi
            energia_phi = funcion_energia(theta_prueba)
            theta_prueba[d] = phi + np.pi / 2
            energia_mas = funcion_energia(theta_prueba)
            theta_prueba[d] = phi - np.pi / 2
            energia_menos = funcion_energia(theta_prueba)

            # Fórmula cerrada que resuelve el mínimo exacto de la onda A*cos + B*sin + C
            theta[d] = phi - np.pi / 2 - np.arctan2(
                2 * energia_phi - energia_mas - energia_menos, energia_mas - energia_menos
            )
    return {"fun": funcion_energia(theta), "params": theta}

## Segundo ejemplo: comparativa completa sobre un Hamiltoniano de dos qubits

Ahora comparas Rotosolve, Nelder-Mead y SPSA sobre un Hamiltoniano de dos qubits con dos términos de Pauli, simulado directamente con Cirq (sin pasar por TensorFlow Quantum) para poder controlar la medición base a base. El ansatz aplica una rotación `rx` y una `rz` por qubit y capa, seguidas de puertas `CNOT` que entrelazan los qubits vecinos.

In [ ]:
import cirq
from scipy.optimize import minimize
from noisyopt import minimizeSPSA

def capa_ansatz(circuito, qubits, parametros):
    for qubit in qubits:
        circuito.append([cirq.rx(parametros[0]).on(qubit)])
        circuito.append([cirq.rz(parametros[1]).on(qubit)])
    for i in range(len(qubits) - 1):
        circuito.append([cirq.CNOT(qubits[i], qubits[i + 1])])
    return circuito

def rotar_a_base_de_medida(circuito, qubits, terminos_pauli):
    for i, qubit in enumerate(qubits):
        if terminos_pauli[i] == "x":
            circuito.append(cirq.ry(-np.pi / 2).on(qubit))
        elif terminos_pauli[i] == "y":
            circuito.append(cirq.rx(np.pi / 2).on(qubit))
    return circuito

def valor_esperado(circuito, qubits_medidos, repeticiones=1000):
    if len(qubits_medidos) == 0:
        return 1
    circuito.append(cirq.measure(*qubits_medidos, key="resultado"))
    simulador = cirq.Simulator()
    resultados = simulador.run(circuito, repetitions=repeticiones)
    medidas = resultados.measurements["resultado"]
    return np.mean([1 if sum(fila) % 2 == 0 else -1 for fila in medidas])

# Hamiltoniano de ejemplo: dos términos de Pauli sobre 2 qubits, con pesos fijos
terminos_hamiltoniano = [["x", "z"], ["z", "y"]]
pesos_hamiltoniano = [0.6, 0.3]
num_qubits, num_capas = 2, 2

def energia(parametros):
    qubits = [cirq.GridQubit(0, i) for i in range(num_qubits)]
    total = 0
    for termino, peso in zip(terminos_hamiltoniano, pesos_hamiltoniano):
        circuito = cirq.Circuit()
        for _ in range(num_capas):
            circuito = capa_ansatz(circuito, qubits, parametros)
        circuito = rotar_a_base_de_medida(circuito, qubits, termino)
        qubits_a_medir = [qubits[i] for i in range(len(termino)) if termino[i] != "i"]
        total += peso * valor_esperado(circuito, qubits_a_medir)
    return total

num_parametros = num_qubits * num_capas

resultado_nm = minimize(energia, np.random.uniform(0, 2 * np.pi, num_parametros), method="Nelder-Mead")
resultado_spsa = minimizeSPSA(energia, bounds=[[0, 2 * np.pi]] * num_parametros,
                               x0=np.random.uniform(0, 2 * np.pi, num_parametros), niter=100, paired=False)
resultado_roto = rotosolve(energia, num_parametros)

print("Nelder-Mead:", resultado_nm["fun"])
print("SPSA:", resultado_spsa["fun"])
print("Rotosolve:", resultado_roto["fun"])

Al ejecutarlo varias veces, comprueba que los tres optimizadores suelen converger a energías parecidas, pero con un número de evaluaciones del circuito muy distinto: Rotosolve suele necesitar menos pasadas completas para estabilizarse, a cambio de evaluar el circuito varias veces por cada parámetro en cada pasada.

## Resumen de puntos clave

- El valor esperado de la energía, al variar un único parámetro de rotación y dejar el resto fijo, siempre tiene la forma de una onda `A·cos(θ) + B·sin(θ) + C`
- Rotosolve aprovecha esa propiedad para calcular el ángulo óptimo exacto de cada parámetro, evaluando el circuito en solo tres puntos por parámetro
- SPSA estima el gradiente completo con solo dos evaluaciones del circuito, sea cual sea el número de parámetros, útil cuando evaluar el circuito es caro
- No existe un optimizador universalmente mejor: la elección depende del número de parámetros, del coste de evaluar el circuito y del ruido presente en esa evaluación